In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed, Bidirectional

from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
import re

# ===========================
# DOWNLOAD NLTK DATA
# ===========================
nltk.download('punkt')
nltk.download('punkt_tab')

# ===========================
# DATASET
# ===========================
sentences = [
    "Ram lives in Pune",
    "She loves programming in Python",
    "The cat sat on the mat",
    "I am learning deep learning",
    "Python is a great language",
    "Machine learning is fascinating",
    "They are playing football",
    "He works at Google",
    "Natural language processing is interesting",
    "She bought a new phone"
]

pos_tags = [
    ["NNP", "VBZ", "IN", "NNP"],
    ["PRP", "VBZ", "VBG", "IN", "NNP"],
    ["DT", "NN", "VBD", "IN", "DT", "NN"],
    ["PRP", "VBP", "VBG", "JJ", "NN"],
    ["NNP", "VBZ", "DT", "JJ", "NN"],
    ["NN", "NN", "VBZ", "JJ"],
    ["PRP", "VBP", "VBG", "NN"],
    ["PRP", "VBZ", "IN", "NNP"],
    ["JJ", "NN", "NN", "VBZ", "JJ"],
    ["PRP", "VBD", "DT", "JJ", "NN"]
]

# ===========================
# PREPROCESSING
# ===========================
def clean_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z\s]', '', sentence)
    return word_tokenize(sentence)

def preprocess(sentences, tags):
    s_out, t_out = [], []
    for i, s in enumerate(sentences):
        words = clean_sentence(s)
        s_out.append(words)
        t_out.append(tags[i])
    return s_out, t_out

# ===========================
# VOCAB
# ===========================
def build_vocab(sentences, tags):
    words = set()
    labels = set()

    for s in sentences:
        words.update(s)
    for t in tags:
        labels.update(t)

    # FIX: +1 indexing
    word_to_idx = {w: i+1 for i, w in enumerate(sorted(words))}
    tag_to_idx = {t: i+1 for i, t in enumerate(sorted(labels))}
    idx_to_tag = {i: t for t, i in tag_to_idx.items()}

    return word_to_idx, tag_to_idx, idx_to_tag

# ===========================
# ENCODING
# ===========================
def encode(sentences, tags, word_to_idx, tag_to_idx):
    X, y = [], []
    max_len = max(len(s) for s in sentences)

    for s, t in zip(sentences, tags):
        X.append([word_to_idx[w] for w in s])
        y.append([tag_to_idx[tag] for tag in t])

    # FIX: padding = 0 (valid label)
    X = pad_sequences(X, maxlen=max_len, padding='post')
    y = pad_sequences(y, maxlen=max_len, padding='post', value=0)

    return X, y, max_len

# ===========================
# MODEL
# ===========================
def build_model(vocab_size, tag_size, max_len):
    model = Sequential([
        Embedding(input_dim=vocab_size+1, output_dim=32),  # removed deprecated input_length
        Bidirectional(LSTM(64, return_sequences=True)),
        Dense(tag_size+1, activation='softmax')  # FIX: +1 for padding class
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ===========================
# PREDICTION
# ===========================
def predict(model, sentence, word_to_idx, idx_to_tag, max_len):
    words = clean_sentence(sentence)

    seq = [word_to_idx.get(w, 0) for w in words]
    padded = pad_sequences([seq], maxlen=max_len, padding='post')

    pred = model.predict(padded, verbose=0)
    pred_tags = np.argmax(pred, axis=-1)[0]

    result = []
    for i, w in enumerate(words):
        tag = idx_to_tag.get(pred_tags[i], "PAD")
        result.append((w, tag))

    return result

# ===========================
# MAIN
# ===========================
if __name__ == "__main__":

    print("\nSTEP 1: PREPROCESSING")
    sentences_p, tags_p = preprocess(sentences, pos_tags)
    print("Sample:", sentences_p[0], tags_p[0])

    print("\nSTEP 2: VOCAB")
    word_to_idx, tag_to_idx, idx_to_tag = build_vocab(sentences_p, tags_p)
    print("Vocab Size:", len(word_to_idx))
    print("Tag Size:", len(tag_to_idx))

    print("\nSTEP 3: ENCODING")
    X, y, max_len = encode(sentences_p, tags_p, word_to_idx, tag_to_idx)
    print("Max Length:", max_len)

    print("\nSTEP 4: SPLIT")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print("\nSTEP 5: MODEL")
    model = build_model(len(word_to_idx), len(tag_to_idx), max_len)
    model.summary()

    print("\nSTEP 6: TRAINING")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,
        batch_size=4,
        verbose=1
    )

    print("\nSTEP 7: EVALUATION")
    loss, acc = model.evaluate(X_test, y_test)
    print(f"Accuracy: {acc*100:.2f}%")

    print("\nSTEP 8: PREDICTIONS")
    test_sentences = [
        "Ram loves coding",
        "The dog ran fast",
        "She works at Microsoft"
    ]

    for s in test_sentences:
        print("\nSentence:", s)
        print("Prediction:", predict(model, s, word_to_idx, idx_to_tag, max_len))

    print("\nSTEP 9: FINAL ACCURACY")
    print(f"Training Accuracy: {history.history['accuracy'][-1]*100:.2f}%")
    print(f"Validation Accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



STEP 1: PREPROCESSING
Sample: ['ram', 'lives', 'in', 'pune'] ['NNP', 'VBZ', 'IN', 'NNP']

STEP 2: VOCAB
Vocab Size: 37
Tag Size: 10

STEP 3: ENCODING
Max Length: 6

STEP 4: SPLIT

STEP 5: MODEL


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


STEP 6: TRAINING
Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 842ms/step - accuracy: 0.1667 - loss: 2.3972 - val_accuracy: 0.1667 - val_loss: 2.3965
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.2500 - loss: 2.3898 - val_accuracy: 0.1667 - val_loss: 2.3918
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.2500 - loss: 2.3822 - val_accuracy: 0.1667 - val_loss: 2.3872
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.2500 - loss: 2.3745 - val_accuracy: 0.1667 - val_loss: 2.3820
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.2500 - loss: 2.3660 - val_accuracy: 0.1667 - val_loss: 2.3763
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.2500 - loss: 2.3564 - val_accuracy: 0.1667 - val_loss: 2.3701
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.2500 - loss: 2.3460 - val_accuracy: 0.1667 - val_loss: 2.3627
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.2500 - loss: 2.3336 - val_accuracy: 0.1